# Blockade-Aware Mapping for H$_2$ on Neutral-Atom Hardware

**Chemistry → Geometry → Noise → Validation** 📐

This notebook extends Notebook 01 by adding a simple hardware-facing geometry layer. The goal is to see how **atom spacing** relative to a **blockade radius** changes the quality and validity of a small quantum chemistry workflow.

Rather than treating the circuit as geometry-free, we introduce a blockade-aware mapping proxy and ask:

- how does spacing affect the effective energy landscape?
- how does spacing interact with noise?
- where do valid execution regions survive?


## 1. Imports

We again use only lightweight scientific Python packages so the notebook remains easy to run and easy to inspect.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


## 2. Minimal H$_2$ Hamiltonian

We keep the same toy 2-qubit H$_2$ Hamiltonian from Notebook 01 so that the new ingredient is clearly the **hardware geometry layer** rather than a change in the chemistry target.


In [ ]:
H2_TERMS = [
    ("II", -1.0523732458),
    ("ZI",  0.3979374248),
    ("IZ", -0.3979374248),
    ("ZZ", -0.0112801043),
    ("XX",  0.1809311998),
]

H2_TERMS


## 3. Pauli operators and energy evaluation

We define the 2-qubit operator machinery needed for expectation values.


In [ ]:
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

PAULI = {
    "I": I,
    "X": X,
    "Y": Y,
    "Z": Z,
}

def kron2(a, b):
    return np.kron(a, b)

def pauli_string_matrix(label: str) -> np.ndarray:
    if len(label) != 2:
        raise ValueError("This notebook expects 2-qubit Pauli strings.")
    return kron2(PAULI[label[0]], PAULI[label[1]])

def hamiltonian_matrix(terms) -> np.ndarray:
    H = np.zeros((4, 4), dtype=complex)
    for label, coeff in terms:
        H = H + coeff * pauli_string_matrix(label)
    return H

def expectation_value(state: np.ndarray, operator: np.ndarray) -> float:
    value = np.vdot(state, operator @ state)
    return float(np.real_if_close(value))

H = hamiltonian_matrix(H2_TERMS)
ground_energy = float(np.min(np.linalg.eigvalsh(H)))

print("Toy Hamiltonian ground-state energy:", ground_energy)
H


## 4. Variational ansatz

We reuse the same small hardware-friendly ansatz from Notebook 01:

1. start from $|00\rangle$
2. apply single-qubit $R_Y$ rotations
3. apply a CNOT entangling gate
4. apply another layer of $R_Y$ rotations

This keeps the comparison clean: the new ingredient in Notebook 02 is **mapping sensitivity to spacing**.


In [ ]:
def ry(theta: float) -> np.ndarray:
    return np.array([
        [np.cos(theta / 2), -np.sin(theta / 2)],
        [np.sin(theta / 2),  np.cos(theta / 2)],
    ], dtype=complex)

def cnot_01() -> np.ndarray:
    return np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
    ], dtype=complex)

def ansatz_state(params) -> np.ndarray:
    theta0, theta1, theta2, theta3 = params

    psi0 = np.array([1, 0, 0, 0], dtype=complex)
    U1 = np.kron(ry(theta0), ry(theta1))
    U2 = np.kron(ry(theta2), ry(theta3))
    Uent = cnot_01()

    psi = U2 @ (Uent @ (U1 @ psi0))
    psi = psi / np.linalg.norm(psi)
    return psi

params_demo = np.array([0.3, 0.8, 0.5, 0.2])
psi_demo = ansatz_state(params_demo)
psi_demo


## 5. Neutral-atom layout and blockade-aware mapping proxy

We now make the geometry layer explicit.

A simple two-atom layout is characterized by:

- number of atoms
- spacing in microns
- blockade radius in microns

When spacing falls below the blockade radius, we treat the configuration as entering a stronger blockade regime. In this notebook, that regime is represented by a simple **mapping penalty** rather than a full pulse-level simulation.

This is intentionally lightweight: the point is to make the chemistry workflow visibly dependent on hardware geometry.


In [ ]:
class NeutralAtomLayout:
    def __init__(self, n_atoms: int, spacing_um: float, blockade_radius_um: float):
        self.n_atoms = n_atoms
        self.spacing_um = spacing_um
        self.blockade_radius_um = blockade_radius_um

    def blockade_active(self) -> bool:
        return self.spacing_um <= self.blockade_radius_um

    def blockade_strength(self) -> float:
        if self.spacing_um >= self.blockade_radius_um:
            return 0.0
        return 1.0 - (self.spacing_um / self.blockade_radius_um)

    def summary(self) -> dict:
        return {
            "n_atoms": self.n_atoms,
            "spacing_um": self.spacing_um,
            "blockade_radius_um": self.blockade_radius_um,
            "blockade_active": self.blockade_active(),
            "blockade_strength": self.blockade_strength(),
        }

layout_demo = NeutralAtomLayout(n_atoms=2, spacing_um=5.5, blockade_radius_um=7.0)
layout_demo.summary()


## 6. Noise model

We keep the same dephasing-style proxy from Notebook 01 so the new effect can be attributed to spacing rather than changing the noise definition.


In [ ]:
REFERENCE_STATE = np.array([1.0, 0.0, 0.0, 0.0], dtype=complex)

def apply_noise_proxy(state: np.ndarray, gamma: float) -> np.ndarray:
    noisy = state.copy()
    noisy[1:] *= np.exp(-gamma)
    norm = np.linalg.norm(noisy)
    if norm == 0:
        raise ValueError("Noisy state norm is zero.")
    return noisy / norm

for g in [0.0, 0.25, 0.5, 1.0]:
    print(g, apply_noise_proxy(psi_demo, g))


## 7. Constraint-based validation 📐

We reuse the overlap-based constraint score from Notebook 01. This lets us see not only how spacing changes the effective energy, but also how it affects the **valid execution region**.


In [ ]:
THRESHOLD = 1.0 / np.sqrt(1.0**2 + 1.0**2)

def constraint_score(state: np.ndarray, reference: np.ndarray = REFERENCE_STATE) -> float:
    denom = np.linalg.norm(state) * np.linalg.norm(reference)
    if denom == 0:
        raise ValueError("Zero norm encountered in constraint_score.")
    value = np.vdot(reference, state) / denom
    return float(np.real_if_close(value))

def passes_constraint(state: np.ndarray, threshold: float = THRESHOLD) -> bool:
    return constraint_score(state) >= threshold

print("Threshold:", THRESHOLD)
print("Constraint score (demo state):", constraint_score(psi_demo))
print("Passes?", passes_constraint(psi_demo))


## 8. Blockade-aware penalties

To make geometry matter, we introduce two simple proxies:

- an **energy penalty** when spacing enters the blockade regime
- an **effective noise increase** when spacing is strongly blockaded

These are intentionally compact engineering proxies, not full physical models. They are enough to make the mapping layer concrete and to produce readable application-facing plots.


In [ ]:
def blockade_energy_penalty(spacing_um: float, blockade_radius_um: float, scale: float = 0.30) -> float:
    if spacing_um >= blockade_radius_um:
        return 0.0
    strength = 1.0 - (spacing_um / blockade_radius_um)
    return scale * strength

def effective_gamma(gamma: float, spacing_um: float, blockade_radius_um: float, scale: float = 0.80) -> float:
    if spacing_um >= blockade_radius_um:
        return gamma
    strength = 1.0 - (spacing_um / blockade_radius_um)
    return gamma + scale * strength

for spacing in [4.0, 6.0, 7.0, 9.0]:
    print(
        spacing,
        "penalty =", blockade_energy_penalty(spacing, 7.0),
        "gamma_eff(0.4) =", effective_gamma(0.4, spacing, 7.0),
    )


## 9. Combined evaluation function

The full application-facing evaluation now depends on:

- variational parameters
- noise level
- atom spacing
- blockade radius

This is the first point in the repo where the chemistry workflow becomes explicitly **geometry-aware**.


In [ ]:
def energy_of_state(state: np.ndarray) -> float:
    return expectation_value(state, H)

def evaluate_params_gamma_spacing(params, gamma: float, spacing_um: float, blockade_radius_um: float) -> dict:
    gamma_eff = effective_gamma(gamma, spacing_um, blockade_radius_um)
    state = ansatz_state(params)
    noisy_state = apply_noise_proxy(state, gamma_eff)

    raw_energy = energy_of_state(noisy_state)
    penalty = blockade_energy_penalty(spacing_um, blockade_radius_um)
    mapped_energy = raw_energy + penalty

    return {
        "params": np.array(params, dtype=float),
        "gamma": float(gamma),
        "gamma_eff": float(gamma_eff),
        "spacing_um": float(spacing_um),
        "blockade_radius_um": float(blockade_radius_um),
        "raw_energy": float(raw_energy),
        "penalty": float(penalty),
        "energy": float(mapped_energy),
        "constraint_score": constraint_score(noisy_state),
        "passes": passes_constraint(noisy_state),
        "state": noisy_state,
    }

demo_result = evaluate_params_gamma_spacing(
    params=params_demo,
    gamma=0.4,
    spacing_um=5.5,
    blockade_radius_um=7.0,
)
demo_result


## 10. Spacing and noise sweep

We now sweep both:

- noise level $\gamma$
- atom spacing

At each pair, we randomly sample ansatz parameters and record:

- best mapped energy over all states
- best mapped energy among constraint-passing states
- whether any valid state exists
- best constraint score

This is the core geometry-aware result of Notebook 02.


In [ ]:
rng = np.random.default_rng(11)

gamma_grid = np.linspace(0.0, 1.25, 50)
spacing_grid = np.linspace(3.0, 10.0, 55)
blockade_radius_um = 7.0
n_samples = 1800

best_energy_all = np.zeros((len(gamma_grid), len(spacing_grid)))
best_energy_valid = np.full((len(gamma_grid), len(spacing_grid)), np.nan)
best_score_all = np.zeros((len(gamma_grid), len(spacing_grid)))
best_score_valid = np.full((len(gamma_grid), len(spacing_grid)), np.nan)
valid_exists = np.zeros((len(gamma_grid), len(spacing_grid)), dtype=bool)

for i, gamma in enumerate(gamma_grid):
    for j, spacing in enumerate(spacing_grid):
        sampled_params = rng.uniform(0.0, 2.0 * np.pi, size=(n_samples, 4))

        energies = []
        scores = []
        passes = []

        for params in sampled_params:
            result = evaluate_params_gamma_spacing(
                params=params,
                gamma=gamma,
                spacing_um=spacing,
                blockade_radius_um=blockade_radius_um,
            )
            energies.append(result["energy"])
            scores.append(result["constraint_score"])
            passes.append(result["passes"])

        energies = np.array(energies)
        scores = np.array(scores)
        passes = np.array(passes, dtype=bool)

        idx_all = int(np.argmin(energies))
        best_energy_all[i, j] = energies[idx_all]
        best_score_all[i, j] = scores[idx_all]

        if np.any(passes):
            valid_energies = np.where(passes, energies, np.inf)
            idx_valid = int(np.argmin(valid_energies))
            best_energy_valid[i, j] = energies[idx_valid]
            best_score_valid[i, j] = scores[idx_valid]
            valid_exists[i, j] = True


## 11. Reference slices at fixed spacing

Before viewing 2D heatmaps, it helps to look at a few representative spacings:

- **strong blockade**: spacing well below the blockade radius
- **near boundary**: spacing near the blockade radius
- **weak blockade / free regime**: spacing above the blockade radius


In [ ]:
spacing_slices = [4.0, 7.0, 9.0]

plt.figure(figsize=(8, 5))
for spacing in spacing_slices:
    j = int(np.argmin(np.abs(spacing_grid - spacing)))
    plt.plot(gamma_grid, best_energy_all[:, j], label=f"Best energy, spacing={spacing_grid[j]:.1f} μm")

plt.axhline(ground_energy, linestyle="--", label="Ground-state energy of toy Hamiltonian")
plt.xlabel("Noise γ")
plt.ylabel("Mapped energy")
plt.title("Best energy vs noise at representative spacings")
plt.legend()
plt.tight_layout()
plt.show()


## 12. Heatmap: best mapped energy over all states

This heatmap shows the best mapped energy found at each spacing/noise pair.

Interpretation:

- lower values are better
- spacing below the blockade radius incurs stronger penalties
- increasing noise also shifts the accessible region


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(
    best_energy_all,
    aspect="auto",
    origin="lower",
    extent=[spacing_grid.min(), spacing_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Best mapped energy (all states)")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Noise γ")
plt.title("Best mapped energy across spacing and noise")
plt.tight_layout()
plt.show()


## 13. Heatmap: best mapped energy among valid states

This heatmap shows the best energy that still passes the validation gate. Missing values indicate that no sampled state satisfied the constraint at that spacing/noise pair.


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(
    best_energy_valid,
    aspect="auto",
    origin="lower",
    extent=[spacing_grid.min(), spacing_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Best mapped energy (constraint-passing)")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Noise γ")
plt.title("Best valid mapped energy across spacing and noise")
plt.tight_layout()
plt.show()


## 14. Heatmap: existence of a valid execution region

This is the most direct geometry-facing validation plot in the notebook. It shows where at least one sampled state survives the constraint gate.


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(
    valid_exists.astype(float),
    aspect="auto",
    origin="lower",
    extent=[spacing_grid.min(), spacing_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Any valid state? (1=yes, 0=no)")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Noise γ")
plt.title("Existence of a valid execution region")
plt.tight_layout()
plt.show()


## 15. Heatmap: best constraint score

This plot helps explain where the validation boundary comes from. Regions with larger best achievable scores are easier to keep above threshold.


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(
    best_score_all,
    aspect="auto",
    origin="lower",
    extent=[spacing_grid.min(), spacing_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Best constraint score (all states)")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Noise γ")
plt.title("Best constraint score across spacing and noise")
plt.tight_layout()
plt.show()


## 16. Derived geometry summary at fixed low noise

As a final summary, we look at a low-noise slice to show how the mapped energy changes as spacing moves from strong blockade into the freer regime above the blockade radius.


In [ ]:
gamma_low = 0.15
i_low = int(np.argmin(np.abs(gamma_grid - gamma_low)))

plt.figure(figsize=(8, 5))
plt.plot(spacing_grid, best_energy_all[i_low], label="Best mapped energy (all states)")
plt.plot(spacing_grid, best_energy_valid[i_low], label="Best mapped energy (constraint-passing)")
plt.axvline(blockade_radius_um, linestyle="--", label="Blockade radius")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Mapped energy")
plt.title(f"Spacing dependence at low noise (γ ≈ {gamma_grid[i_low]:.2f})")
plt.legend()
plt.tight_layout()
plt.show()


## 17. Save a figure for the repo

This cell saves a clean geometry-facing figure to `../figures/` when run from the `notebooks/` directory inside the repo. The directory is created automatically if needed.


In [ ]:
output_dir = os.path.join("..", "figures")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "blockade_mapping_valid_region.png")

plt.figure(figsize=(8, 5))
plt.imshow(
    valid_exists.astype(float),
    aspect="auto",
    origin="lower",
    extent=[spacing_grid.min(), spacing_grid.max(), gamma_grid.min(), gamma_grid.max()],
)
plt.colorbar(label="Any valid state? (1=yes, 0=no)")
plt.xlabel("Atom spacing (μm)")
plt.ylabel("Noise γ")
plt.title("Existence of a valid execution region")
plt.tight_layout()
plt.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {output_path}")


## 18. Summary

Notebook 02 extends the first chemistry workflow by adding an explicit **blockade-aware mapping layer**.

The main idea is simple:

- below the blockade radius, geometry pushes the system into a more constrained regime
- this changes the effective energy landscape
- noise and geometry together determine whether valid execution survives

This notebook is still intentionally lightweight, but it makes the repo meaningfully more **neutral-atom specific**.


## Notes

This workflow can also be interpreted as a three-stage pipeline:

- chemistry target
- geometry-aware mapping
- execution validation

That gives the repository a more hardware-facing applications story than Notebook 01 alone.
